<a href="https://colab.research.google.com/github/anelisesrocha/Projeto-Integrador/blob/main/Teste_c%C3%B3digo_(Padr%C3%A3o).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch scikit-learn pandas

import pandas as pd
import re
from transformers import BertTokenizer, BertModel
import torch
from sklearn.cluster import KMeans

# 1. Carregar base (suba o CSV tratado antes)
df = pd.read_csv("manchetes_google_news_limpo.csv", encoding="utf-8-sig", sep=",")

# 2. Preparar BERT em português
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
model = BertModel.from_pretrained("neuralmind/bert-base-portuguese-cased")

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=64)
    outputs = model(**inputs)
    return outputs.last_hidden_state[:,0,:].detach().numpy().flatten()

# 3. Regras semânticas inspiradas em Palau-Sampio (cap. 5)
def detect_patterns(headline):
    patterns = []
    h = headline.lower()

    # Perguntas retóricas
    if "?" in h:
        patterns.append("Pergunta retórica")

    # Exagero lexical
    if re.search(r"(incrível|chocante|surpreendente|jamais|nunca|sempre|todos|melhor|pior|único|fantástico)", h):
        patterns.append("Exagero lexical")

    # Suspense/lacuna
    if re.search(r"(você não vai acreditar|descubra|saiba mais|o segredo|revelado|mistério)", h):
        patterns.append("Suspense/lacuna")

    # Referência vaga
    if re.search(r"(este|essa|aquele|um estudo|um médico|especialistas|pesquisadores)", h):
        patterns.append("Referência vaga")

    # Apelo emocional/sensacionalista
    if re.search(r"(emocionante|comovente|triste|feliz|chocante|polêmico|escândalo|dramático)", h):
        patterns.append("Apelo emocional")

    # Apelo à curiosidade/listas
    if re.search(r"(veja|confira|lista|top \d+|ranking|os \d+ principais)", h):
        patterns.append("Apelo à curiosidade")

    # Uso de tempo verbal urgente
    if re.search(r"(agora|já|imediatamente|urgente)", h):
        patterns.append("Urgência")

    # Uso de números para atrair atenção
    if re.search(r"\d+", h):
        patterns.append("Uso de números")

    return patterns if patterns else ["Nenhum padrão"]

# 4. Gerar embeddings e padrões
embeddings = []
results = []
for idx, row in df.iterrows():
    manchete = row["Manchete"]
    emb = get_bert_embedding(manchete)
    embeddings.append(emb)
    pats = detect_patterns(manchete)
    results.append({
        "Manchete": manchete,
        "Padrões_detectados": "; ".join(pats)
    })

# 5. Clustering com KMeans
X = embeddings
n_clusters = 6  # ajuste conforme desejar
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(X)

for i, r in enumerate(results):
    r["Cluster"] = labels[i]

# 6. Exportar resultados
out_df = pd.DataFrame(results)
out_df.to_csv("manchetes_clickbait_clusters.csv", index=False, encoding="utf-8-sig")

print("✅ Arquivo 'manchetes_clickbait_clusters.csv' gerado.")
print(out_df.head(10))


ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            

KeyboardInterrupt: 

In [ ]:
def limpar_manchete(texto):
    texto = str(texto).strip()

    # Remover domínio ou veículo no final (ex: " - Globo", " - Estadão", " - saude.ba.gov.br")
    texto = re.sub(r"\s*-\s*(globo|estadao|folha|g1|gov\.br|uol|terra|r7|cnn|bbc|poder360|agência brasil|[\w\.]+\.br)$", "", texto, flags=re.IGNORECASE)

    # Remover URLs misturadas
    texto = re.sub(r"http\S+", "", texto)

    # Remover espaços extras
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


In [ ]:
df["Manchete_limpa"] = df["Manchete"].apply(limpar_manchete)


In [ ]:
!pip install transformers torch scikit-learn pandas

import pandas as pd
import re
from transformers import BertTokenizer, BertModel
import torch
from sklearn.cluster import KMeans

# 1. Carregar base tratada
df = pd.read_csv("manchetes_google_news_limpo.csv", encoding="utf-8-sig", sep=",")

# 2. Limpar manchetes (remover nomes de veículos misturados)
def limpar_manchete(texto):
    texto = str(texto).strip()
    texto = re.sub(r"\s*-\s*(globo|estadao|folha|g1|gov\.br|uol|terra|r7|cnn|bbc|poder360|agência brasil|[\w\.]+\.br)$", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"http\S+", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["Manchete_limpa"] = df["Manchete"].apply(limpar_manchete)

# 3. Preparar BERT
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
model = BertModel.from_pretrained("neuralmind/bert-base-portuguese-cased")

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=32)
    outputs = model(**inputs)
    return outputs.last_hidden_state[:,0,:].detach().numpy().flatten()

# 4. Regras semânticas Palau-Sampio
def detectar_padroes(texto):
    texto = texto.lower()
    padroes = []

    if "?" in texto:
        padroes.append("Pergunta retórica")
    if re.search(r"(incrível|chocante|surpreendente|jamais|nunca|sempre|todos|melhor|pior|único|fantástico)", texto):
        padroes.append("Exagero lexical")
    if re.search(r"(você não vai acreditar|descubra|saiba mais|o segredo|revelado|mistério)", texto):
        padroes.append("Suspense/lacuna")
    if re.search(r"(este|essa|aquele|um estudo|um médico|especialistas|pesquisadores)", texto):
        padroes.append("Referência vaga")
    if re.search(r"(emocionante|comovente|triste|feliz|chocante|polêmico|escândalo|dramático)", texto):
        padroes.append("Apelo emocional")
    if re.search(r"(veja|confira|lista|top \d+|ranking|os \d+ principais)", texto):
        padroes.append("Apelo à curiosidade")
    if re.search(r"(agora|já|imediatamente|urgente)", texto):
        padroes.append("Urgência")
    if re.search(r"\d+", texto):
        padroes.append("Uso de números")

    return "; ".join(padroes) if padroes else "Nenhum padrão"

# 5. Gerar embeddings e padrões
embeddings = []
padroes = []

for manchete in df["Manchete_limpa"]:
    emb = get_bert_embedding(manchete)
    embeddings.append(emb)
    padroes.append(detectar_padroes(manchete))

# 6. Clustering com KMeans
n_clusters = 6
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

# 7. Adicionar colunas ao df original
df["Padrões_detectados"] = padroes
df["Cluster"] = labels

# 8. Exportar resultado
df.to_csv("manchetes_google_news_com_clusters.csv", index=False, encoding="utf-8-sig")

print("✅ Arquivo 'manchetes_google_news_com_clusters.csv' gerado com todas as colunas originais + padrões + cluster.")
print(df[["Manchete", "Padrões_detectados", "Cluster"]].head(10))


✅ Arquivo 'manchetes_google_news_com_clusters.csv' gerado com todas as colunas originais + padrões + cluster.
                                            Manchete   Padrões_detectados  \
0  Por que cada vez mais estrangeiros estão vindo...    Pergunta retórica   
1  Plano de saúde dos Correios será completamente...       Uso de números   
2  Superidosos dão 5 dicas para viver mais e com ...       Uso de números   
3  Ministério da Saúde lança Chamadas Públicas pa...        Nenhum padrão   
4  Saúde lança versão atualizada da Caderneta da ...        Nenhum padrão   
5  Vida saudável em 2026: o que a ciência diz sob...       Uso de números   
6  Confira o funcionamento das unidades de saúde ...  Apelo à curiosidade   
7  Janeiro Branco: campanha nacional sobre saúde ...        Nenhum padrão   
8  Judicialização dos planos de saúde deve acompa...        Nenhum padrão   
9  Ministério da Saúde envia equipe para monitora...        Nenhum padrão   

   Cluster  
0        0  
1        5  
2  

In [ ]:
!pip install transformers torch scikit-learn pandas

import pandas as pd
import re
from transformers import BertTokenizer, BertModel
import torch
from sklearn.cluster import KMeans

# 1. Carregar base tratada
df = pd.read_csv("manchetes_google_news_limpo.csv", encoding="utf-8-sig", sep=",")

# 2. Limpar manchetes (remover nomes de veículos misturados)
def limpar_manchete(texto):
    texto = str(texto).strip()
    texto = re.sub(r"\s*-\s*(globo|estadao|folha|g1|gov\.br|uol|terra|r7|cnn|bbc|poder360|agência brasil|[\w\.]+\.br)$", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"http\S+", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["Manchete_limpa"] = df["Manchete"].apply(limpar_manchete)

# 3. Preparar BERT
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
model = BertModel.from_pretrained("neuralmind/bert-base-portuguese-cased")

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=32)
    outputs = model(**inputs)
    return outputs.last_hidden_state[:,0,:].detach().numpy().flatten()

# 4. Regras semânticas Palau-Sampio
def detectar_padroes(texto):
    texto = texto.lower()
    padroes = []

    if "?" in texto:
        padroes.append("Pergunta retórica")
    if re.search(r"(incrível|chocante|surpreendente|jamais|nunca|sempre|todos|melhor|pior|único|fantástico)", texto):
        padroes.append("Exagero lexical")
    if re.search(r"(você não vai acreditar|descubra|saiba mais|o segredo|revelado|mistério)", texto):
        padroes.append("Suspense/lacuna")
    if re.search(r"(este|essa|aquele|um estudo|um médico|especialistas|pesquisadores)", texto):
        padroes.append("Referência vaga")
    if re.search(r"(emocionante|comovente|triste|feliz|chocante|polêmico|escândalo|dramático)", texto):
        padroes.append("Apelo emocional")
    if re.search(r"(veja|confira|lista|top \d+|ranking|os \d+ principais)", texto):
        padroes.append("Apelo à curiosidade")
    if re.search(r"(agora|já|imediatamente|urgente)", texto):
        padroes.append("Urgência")
    if re.search(r"\d+", texto):
        padroes.append("Uso de números")

    return "; ".join(padroes) if padroes else "Nenhum padrão"

# 5. Gerar embeddings e padrões
embeddings = []
padroes = []

for manchete in df["Manchete_limpa"]:
    emb = get_bert_embedding(manchete)
    embeddings.append(emb)
    padroes.append(detectar_padroes(manchete))

# 6. Clustering com KMeans
n_clusters = 6
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(embeddings)

# 7. Adicionar colunas ao df original
df["Padrões_detectados"] = padroes
df["Cluster"] = labels

# 8. Exportar resultado
df.to_csv("manchetes_google_news_com_clusters.csv", index=False, encoding="utf-8-sig")

print("✅ Arquivo 'manchetes_google_news_com_clusters.csv' gerado com todas as colunas originais + padrões + cluster.")
print(df[["Manchete", "Padrões_detectados", "Cluster"]].head(10))


In [ ]:
# Definir pesos heurísticos
pesos = {
    "Pergunta retórica": 0.25,
    "Exagero lexical": 0.20,
    "Suspense/lacuna": 0.20,
    "Apelo emocional": 0.15,
    "Apelo à curiosidade": 0.10,
    "Urgência": 0.05,
    "Uso de números": 0.05
}

def calcular_clickbait_score(padroes_str):
    padroes = padroes_str.split("; ")
    score = sum(pesos.get(p, 0) for p in padroes)
    return min(score, 1.0) * 100  # em %

# Aplicar às manchetes
df["Clickbait_%"] = df["Padrões_detectados"].apply(calcular_clickbait_score)

# Exportar resultado
df.to_csv("manchetes_clickbait_score.csv", index=False, encoding="utf-8-sig")

print(df[["Manchete","Padrões_detectados","Clickbait_%"]].head(10))


                                            Manchete   Padrões_detectados  \
0  Por que cada vez mais estrangeiros estão vindo...    Pergunta retórica   
1  Plano de saúde dos Correios será completamente...       Uso de números   
2  Superidosos dão 5 dicas para viver mais e com ...       Uso de números   
3  Ministério da Saúde lança Chamadas Públicas pa...        Nenhum padrão   
4  Saúde lança versão atualizada da Caderneta da ...        Nenhum padrão   
5  Vida saudável em 2026: o que a ciência diz sob...       Uso de números   
6  Confira o funcionamento das unidades de saúde ...  Apelo à curiosidade   
7  Janeiro Branco: campanha nacional sobre saúde ...        Nenhum padrão   
8  Judicialização dos planos de saúde deve acompa...        Nenhum padrão   
9  Ministério da Saúde envia equipe para monitora...        Nenhum padrão   

   Clickbait_%  
0         25.0  
1          5.0  
2          5.0  
3          0.0  
4          0.0  
5          5.0  
6         10.0  
7          0.0  

In [ ]:
from google.colab import files
files.download("manchetes_clickbait_score.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
Definir pesos heurísticos
pesos = {
    "Pergunta retórica": 0.25,
    "Exagero lexical": 0.20,
    "Suspense/lacuna": 0.20,
    "Apelo emocional": 0.15,
    "Apelo à curiosidade": 0.10,
    "Urgência": 0.05,
    "Uso de números": 0.05
}

def calcular_clickbait_score(padroes_str):
    padroes = padroes_str.split("; ")
    score = sum(pesos.get(p, 0) for p in padroes)
    return min(score, 1.0) * 100  # em %

# Aplicar às manchetes
df["Clickbait_%"] = df["Padrões_detectados"].apply(calcular_clickbait_score)

# Exportar resultado
df.to_csv("manchetes_clickbait_score.csv", index=False, encoding="utf-8-sig")

print(df[["Manchete","Padrões_detectados","Clickbait_%"]].head(10))


SyntaxError: invalid syntax (3393237060.py, line 1)